In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import librosa
import logging
import torch
import torchaudio
from tqdm import tqdm
from transformers import Wav2Vec2Processor, Wav2Vec2Model

from src.helper import *
from src.embedding import *

In [2]:
#set config
DATA_PATH = "data/deep-detect/dataset/"
OUTPUT_PATH = "output/"
EMBEDDING_PATH = "output/embeddings/"
MODEL_NAME = "facebook/wav2vec2-base"
SAMPLE_RATE = 16000

In [3]:
if not os.path.exists(OUTPUT_PATH):
    os.mkdir(OUTPUT_PATH)
if not os.path.exists(EMBEDDING_PATH):
    os.mkdir(EMBEDDING_PATH)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
# --- Setup logging ---
logging.basicConfig(
    filename=os.path.join(OUTPUT_PATH, f"nb_03_dl_feature_engineering_log.log"),
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger("Main")
logger.info(f"starting DL feature engineering...")

### Applying wav2vec2 to extract embeddings

In [6]:
# ----------------------------
# Load model & processor
# ----------------------------
logger.info(f"loading {MODEL_NAME} pretrained processor...")
processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
model = Wav2Vec2Model.from_pretrained(MODEL_NAME).to(device)
model.eval()

/home/ardacandra/miniconda3/envs/deepdetect_audio_deepfake_detection_challenge/lib/python3.13/site-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


Wav2Vec2Model(
  (feature_extractor): Wav2Vec2FeatureEncoder(
    (conv_layers): ModuleList(
      (0): Wav2Vec2GroupNormConvLayer(
        (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
        (activation): GELUActivation()
        (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
      )
      (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
      (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
    )
  )
  (feature_projection): Wav2Vec2FeatureProjection(
    (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (projection): Linear(in_features=512, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): Wav2Vec2Encoder(
    (pos_conv_embed): Wav2Vec2PositionalConvEmbedding(
  

In [7]:
wav2vec2_embedding_path = os.path.join(EMBEDDING_PATH, "wav2vec2/")
if not os.path.exists(wav2vec2_embedding_path):
    os.mkdir(wav2vec2_embedding_path)

In [ ]:
logger.info(f"starting {MODEL_NAME} embedding for all audio files ...")
logger.info(f"saving result to {wav2vec2_embedding_path}")
for split in ["holdout", "testing", "training"]:

    split_output_dir = os.path.join(wav2vec2_embedding_path, split)
    if not os.path.exists(split_output_dir):
        os.mkdir(split_output_dir)
        
    for label in ["", "real", "fake"]:

        folder = os.path.join(DATA_PATH, split, label)
        if not os.path.exists(folder):
            continue

        label_output_dir = os.path.join(split_output_dir, label)
        if not os.path.exists(label_output_dir):
            os.mkdir(label_output_dir)

        for fname in os.listdir(folder):
            fpath = os.path.join(folder, fname)
            if os.path.isfile(fpath):
                try:
                    #get embedding
                    embedding = extract_wav2vec2_embedding(
                        fpath,
                        SAMPLE_RATE,
                        processor,
                        device,
                        model
                    )
                    #save as .npy
                    np.save(os.path.join(label_output_dir, f"{fname.split('.')[0]}.npy"), embedding)
                    logger.info(f"finished processing {fname}")

                except Exception as e:
                    logger.info(f"unable to process {fname} : {e}")
                    np.save(os.path.join(label_output_dir, f"{fname.split('.')[0]}.npy"), np.zeros(model.config.hidden_size, dtype=np.float32))
                    logger.info(f"created placeholder embedding for {fname}")

logger.info(f"finished processing all audio files")

/home/ardacandra/miniconda3/envs/deepdetect_audio_deepfake_detection_challenge/lib/python3.13/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
